# Phase 1: Spurgeon Continued Pretraining — Step 6: Dataset Preparation (Notebook A)

This notebook loads the cleaned Spurgeon train and holdout text datasets, parses them, converts them to Hugging Face `Dataset` structures, splits the training set into train/validation splits, and saves them to disk so they can be exported as Kaggle input datasets for training.

## 1. Install Dependencies

In [1]:
# Install lightweight Hugging Face datasets library
!pip install datasets -q

## 2. Load and Parse Training Corpus

In [2]:
import os
from pathlib import Path
from datasets import Dataset

train_corpus_path = "/kaggle/input/datasets/rafaelvieira1/spurgeon-cpt-corpus/spurgeon_train.txt"

if not os.path.exists(train_corpus_path):
    raise FileNotFoundError(f"Training corpus file not found at: {train_corpus_path}. "
                            "Please ensure the 'spurgeon-cpt-corpus' dataset is added to this notebook.")

print(f"Loading training corpus from {train_corpus_path}...")
with open(train_corpus_path, "r", encoding="utf-8") as f:
    train_text = f.read()

print(f"Corpus size: {len(train_text):,} characters.")

# Split into separate documents at sermon boundaries
raw_docs = train_text.split('<|endoftext|>')
# Filter out empty documents, trailing spaces, or stubs under 200 characters
train_docs = [doc.strip() for doc in raw_docs if len(doc.strip()) > 200]

print(f"Parsed {len(train_docs)} training documents (skipped {len(raw_docs) - len(train_docs)} short segments/stubs).")

# Instantiate Hugging Face Dataset
train_dataset = Dataset.from_dict({"text": train_docs})
print("Hugging Face Dataset successfully created:")
print(train_dataset)

Loading training corpus from /kaggle/input/datasets/rafaelvieira1/spurgeon-cpt-corpus/spurgeon_train.txt...
Corpus size: 127,584,192 characters.
Parsed 3486 training documents (skipped 1 short segments/stubs).
Hugging Face Dataset successfully created:
Dataset({
    features: ['text'],
    num_rows: 3486
})


## 3. Create Train-Validation Split

In [3]:
dataset_split = train_dataset.train_test_split(test_size=0.01, seed=42)

print("Split Results:")
print(f"  - Train split: {len(dataset_split['train'])} documents")
print(f"  - Validation split: {len(dataset_split['test'])} documents")

Split Results:
  - Train split: 3451 documents
  - Validation split: 35 documents


## 4. Load and Parse Holdout Corpus

In [4]:
holdout_corpus_path = "/kaggle/input/datasets/rafaelvieira1/spurgeon-cpt-holdout/spurgeon_holdout.txt"

if not os.path.exists(holdout_corpus_path):
    raise FileNotFoundError(f"Holdout corpus file not found at: {holdout_corpus_path}. "
                            "Please ensure the 'spurgeon-cpt-holdout' dataset is added to this notebook.")

print(f"Loading holdout corpus from {holdout_corpus_path}...")
with open(holdout_corpus_path, "r", encoding="utf-8") as f:
    holdout_text = f.read()

print(f"Holdout corpus size: {len(holdout_text):,} characters.")

# Split into separate documents at sermon boundaries
raw_holdout_docs = holdout_text.split('<|endoftext|>')
holdout_docs = [doc.strip() for doc in raw_holdout_docs if len(doc.strip()) > 200]

print(f"Parsed {len(holdout_docs)} holdout documents (skipped {len(raw_holdout_docs) - len(holdout_docs)} segments).")

# Create holdout Hugging Face Dataset
holdout_dataset = Dataset.from_dict({"text": holdout_docs})
print(holdout_dataset)

Loading holdout corpus from /kaggle/input/datasets/rafaelvieira1/spurgeon-cpt-holdout/spurgeon_holdout.txt...
Holdout corpus size: 1,829,424 characters.
Parsed 50 holdout documents (skipped 1 segments).
Dataset({
    features: ['text'],
    num_rows: 50
})


## 5. Save Datasets and Verify

In [5]:
train_save_path = "/kaggle/working/spurgeon_dataset"
holdout_save_path = "/kaggle/working/spurgeon_holdout_dataset"

print(f"Saving training/validation dataset split to {train_save_path}...")
dataset_split.save_to_disk(train_save_path)

print(f"Saving holdout dataset to {holdout_save_path}...")
holdout_dataset.save_to_disk(holdout_save_path)

# Verify directory structures and file outputs on disk
print("\nVerifying files in /kaggle/working/:")
for root, dirs, files in os.walk("/kaggle/working/"):
    depth = root.replace("/kaggle/working/", "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}[D] {os.path.basename(root) or 'working/'}")
    for file in files:
        file_path = os.path.join(root, file)
        size_kb = os.path.getsize(file_path) / 1024
        print(f"{indent}  - {file} ({size_kb:.2f} KB)")
        
print("\nDataset preparation completed successfully!")

Saving training/validation dataset split to /kaggle/working/spurgeon_dataset...


Saving the dataset (0/1 shards):   0%|          | 0/3451 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/35 [00:00<?, ? examples/s]

Saving holdout dataset to /kaggle/working/spurgeon_holdout_dataset...


Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]


Verifying files in /kaggle/working/:
[D] working/
  - __notebook__.ipynb (11.57 KB)
[D] spurgeon_dataset
  - dataset_dict.json (0.03 KB)
  [D] test
    - data-00000-of-00001.arrow (1195.17 KB)
    - state.json (0.24 KB)
    - dataset_info.json (0.16 KB)
  [D] train
    - data-00000-of-00001.arrow (123379.97 KB)
    - state.json (0.24 KB)
    - dataset_info.json (0.16 KB)
[D] spurgeon_holdout_dataset
  - data-00000-of-00001.arrow (1786.55 KB)
  - state.json (0.24 KB)
  - dataset_info.json (0.16 KB)

Dataset preparation completed successfully!
